# 5. PDF Onboarding (Aryn)

**Goal:** Enable the agent to answer compliance questions from unstructured PDF documents.

**Key Concept:**
We utilize the **Aryn SDK** (DataRobot's partner for document intelligence) to intelligently parse and chunk a PDF (e.g., *Supplier Quality Standards*). This enables a RAG (Retrieval Augmented Generation) workflow, allowing the agent to read the policy and verify if a delivery (e.g., "Butter at 6°C") should be accepted or rejected.

In [7]:
import os
from aryn_sdk.partition import partition_file

# --- CONFIGURATION ---
PDF_FILENAME = "archive/ercot_market_briefing.pdf"

# 1. Define Fallback Policy (Simulation Data)
# This ensures the demo works 100% of the time, even if the API key is missing.
fallback_policy = """
OFFICIAL INGREDIENT HANDLING POLICY (v2025.1)
1. FLOUR: Must be stored between 15°C and 25°C. Moisture content cannot exceed 14%.
2. BUTTER: Deliveries must be received at 4°C or below. Rejection threshold is 7°C.
3. YEAST: Fresh yeast must be used within 7 days of delivery.
4. HONEY: Only Grade A filtered honey is accepted. Crystallized honey must be returned.
"""

# 2. Parse Document
try:
    # We check if the file exists. 
    # aryn-sdk will automatically check os.environ["ARYN_API_KEY"] when we call partition_file
    if os.path.exists(PDF_FILENAME):
        print(f"🚀 Found {PDF_FILENAME}. Attempting to parse with Aryn...")
        
        with open(PDF_FILENAME, "rb") as f:
            # CLEAN CALL: No API key passed here; SDK looks for 'ARYN_API_KEY' in env vars
            data = partition_file(f, use_ocr=True, extract_table_structure=True)
            
        # Extract plain text from the structured response
        if isinstance(data, dict) and 'elements' in data:
            context_text = "\n".join([e.get('text', '') for e in data['elements'] if e.get('text')])
        else:
            context_text = str(data)
            
        print(f"✅ Success! Extracted {len(context_text)} characters.")

    else:
        print(f"⚠️ {PDF_FILENAME} not found. Using simulation data.")
        context_text = fallback_policy

except Exception as e:
    # If the ARYN_API_KEY env var is missing, this block catches the error and keeps the demo alive
    print(f"ℹ️ Aryn Parsing skipped: {e}")
    print("   (This is expected if ARYN_API_KEY is not set in your Environment Variables)")
    print("   -> Switching to Simulation Mode.")
    context_text = fallback_policy

print("\n--- EXTRACTED CONTEXT ---")
print(context_text[:500] + "...")

🚀 Found archive/ercot_market_briefing.pdf. Attempting to parse with Aryn...
ℹ️ Aryn Parsing skipped: Could not find an aryn api key. Checked the ARYN_API_KEY env var, the /home/notebooks/.aryn/config.yaml config file, and the aryn_api_key parameter
   (This is expected if ARYN_API_KEY is not set in your Environment Variables)
   -> Switching to Simulation Mode.

--- EXTRACTED CONTEXT ---

OFFICIAL INGREDIENT HANDLING POLICY (v2025.1)
1. FLOUR: Must be stored between 15°C and 25°C. Moisture content cannot exceed 14%.
2. BUTTER: Deliveries must be received at 4°C or below. Rejection threshold is 7°C.
3. YEAST: Fresh yeast must be used within 7 days of delivery.
4. HONEY: Only Grade A filtered honey is accepted. Crystallized honey must be returned.
...


In [5]:
import os
from aryn_sdk.partition import partition_file

# --- CONFIGURATION ---
PDF_FILENAME = "archive/ercot_market_briefing.pdf"

# 1. Define fallback text (Simulation Mode)
# This ensures the notebook runs smoothly during a demo even if the file isn't uploaded.
fallback_policy = """
OFFICIAL INGREDIENT HANDLING POLICY (v2025.1)
1. FLOUR: Must be stored between 15°C and 25°C. Moisture content cannot exceed 14%.
2. BUTTER: Deliveries must be received at 4°C or below. Rejection threshold is 7°C.
3. YEAST: Fresh yeast must be used within 7 days of delivery.
4. HONEY: Only Grade A filtered honey is accepted. Crystallized honey must be returned.
"""

# 2. Use Aryn to Partition the PDF
try:
    if os.path.exists(PDF_FILENAME):
        print(f"Found {PDF_FILENAME}. Parsing with Aryn...")
        with open(PDF_FILENAME, 'rb') as f:
            # partition_file extracts text, tables, and layout info
            elements = partition_file(f)
            
        # Extract clear text from the elements
        context_text = "\n".join([e.text for e in elements if e.text])
        print(f"✅ Successfully extracted {len(context_text)} characters from PDF.")
        
    else:
        print(f"⚠️ {PDF_FILENAME} not found. Using simulation data for this demo.")
        context_text = fallback_policy

except Exception as e:
    print(f"Warning: PDF processing failed ({e}). Falling back to simulation data.")
    context_text = fallback_policy

print("\n--- EXTRACTED CONTEXT ---")
print(context_text[:500] + "...") # Preview the first 500 chars

Found archive/ercot_market_briefing.pdf. Parsing with Aryn...

--- EXTRACTED CONTEXT ---

OFFICIAL INGREDIENT HANDLING POLICY (v2025.1)
1. FLOUR: Must be stored between 15°C and 25°C. Moisture content cannot exceed 14%.
2. BUTTER: Deliveries must be received at 4°C or below. Rejection threshold is 7°C.
3. YEAST: Fresh yeast must be used within 7 days of delivery.
4. HONEY: Only Grade A filtered honey is accepted. Crystallized honey must be returned.
...
